In [1]:
from dotenv import load_dotenv

In [2]:
_env = '../.env'

In [3]:
load_dotenv(_env)

True

In [4]:
import os
api_key = os.environ.get("OPENAI_API_KEY")

In [5]:
from openai import OpenAI

In [6]:
client = OpenAI(api_key= api_key)

In [7]:
JUDGE_PROMPT="""
You are a judge LLM. Please judge the agent response against reference response based on the following
criteria:
- correctness (0 or 1 binary)
- helpfulness (0 or 1 binary)
- reasoning (string with 1-2 sentence justifying the score)
Make sure you have output only in JSON format with those 3 fields.
"""

In [22]:
def llm_as_judge(user_prompt, system_prompt):
    resp = client.chat.completions.create(
        model='gpt-5.4-mini',
        temperature=0,
        messages=[
            {'role':'system', 'content':system_prompt},
            {'role':'user', 'content':user_prompt}
            ]
        )
    return resp.choices[0].message.content


In [9]:
def build_user_prompt(question, prediction, reference):
    return (
        f'Question: {question}\n'
        f'Agent response: {prediction}\n'
        f'Reference resposne: {reference}\n'
        'Your judge output should only be in JSON.'
    )

In [10]:
from datasets import load_dataset
ds = load_dataset("rajpurkar/squad_v2")

/Users/Larry.Jin/miniconda3/envs/prep/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating validation split: 100%|████| 11873/11873 [00:00<00:00, 1166279.57 examples/s]


In [11]:
val_ds =ds['validation']

In [14]:
# copy-pasted
def squad_reference_text(row: dict) -> str:
    texts = [t.strip() for t in row["answers"]["text"] if (t or "").strip()]
    return texts[0] if texts else ""


def is_answerable_row(row: dict) -> bool:
    return bool(squad_reference_text(row))


answerable_val = val_ds.filter(is_answerable_row)
subset = answerable_val.shuffle(seed=42).select(range(20))
subset

Filter: 100%|██████████████████████████| 11873/11873 [00:00<00:00, 156419.53 examples/s]


Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 20
})

In [18]:
QA_SYSTEM = (
    "You answer reading-comprehension questions using only the passage. "
    "Reply with the shortest correct answer phrase or sentence. No preamble, no quotes."
)


def generate_answer(context: str, question: str) -> str:
    user = f"Passage:\n{context}\n\nQuestion:\n{question}"
    resp = client.chat.completions.create(
        model='gpt-5.4-mini',
        temperature=0,
        messages=[
            {"role": "system", "content": QA_SYSTEM},
            {"role": "user", "content": user},
        ],
    )
    return (resp.choices[0].message.content or "").strip()


predictions: list[str] = []
references: list[str] = []
meta: list[dict] = []

for i, row in enumerate(subset):
    ref = squad_reference_text(row)
    pred = generate_answer(row["context"], row["question"])
    predictions.append(pred)
    references.append(ref)
    meta.append({"id": row["id"], "question": row["question"], "reference": ref, "prediction": pred})
    # time.sleep(0.15)  # light pacing for rate limits

meta[:2]

[{'id': '573020f7b2c2fd14005688fa',
  'question': 'When did Hamas drive the PLO out of Gaza?',
  'reference': '2007',
  'prediction': '2007'},
 {'id': '57335ddbd058e614000b5932',
  'question': 'Where can Aeolian sand with a number of dunes be found?',
  'reference': 'plain Vistula terraces',
  'prediction': 'On the highest terrace on the right side of Warsaw.'}]

In [29]:
import json
result = []
for item in meta:
    q = item['question']
    p = item['prediction']
    r = item['reference']
    prompt = build_user_prompt(q, p, r)
    res = llm_as_judge(prompt, JUDGE_PROMPT)
    result.append(json.loads(res.strip()))

In [28]:
result[0]['correctness']

1